# PCA - Principal component analysis (метод главных компонент)

Одна из главных проблем ЭЭГ - каждый электрод пишет сигнал разного происхождения, включая тот "шум", который нам не нужен. Исследователям очень хотелось бы, чтобы электрод, расположенный, над прецентральной извилиной, записывал изменения потенциала только её пирамидных нейронов. Но, к сожалению, моргание, сокращение мышц (сигналы высокой амплитуды), активность соседних зон коры - в каждый электрод попадает всё и сразу. И выделить "полезный" сигнал, отбросив "моргания и чихания" помогает математика. В частности, PCA - метод главных компонент. 

----------------------------------------------------------------------------------------------------------------------------------------------

Представьте, что у вас есть два канала (электрода) - channel_1 и channel_2. На каждом вы сделали по 4 замера разности потенциалов (в моменты времени t1, t2, t3, t4). Каждый электрод поймал часть сигнала в области альфа-ритма (то, что мы хотим исследовать) и часть высокоамплитудного сигнала от моргания человека в момент эксперимента. Нам нужно выкинуть высокоамплитудный компонент моргания и оставить альфа-ритм. 

Наш сигнал выглядит следующим образом:

<div style="display: flex; justify-content: center; gap: 20px;">
    <div style="text-align: center;">
        <a href="assets/Plot_PCA_without_axes.png">
            <img src="assets/Plot_PCA_without_axes.png" width="500" style="max-width: none;">
            <br>
            <b>Рис. 1.</b> Каждая точка - один момент времени 
            (координата x - напряжение на первом электроде, координата y - напряжение на втором электроде)
            <a href="assets/Plot_PCA_without_axes.png">
        </a>
    </div>
    <div style="text-align: center;">
        <a href="assets/Plot_PCA_axes.png">
            <img src="assets/Plot_PCA_axes.png" width="500" style="max-width: none;">
            <br>
            <b>Рис. 2.</b> Векторы, в направлении которых разбросаны наши точки наибольшим образом (можно сказать, что наши точки образуют форму эллипса, а векторы - оси этого эллипса)
    </div>
</div>


Обе линии на втором графике - векторы, на которые мы планируем спроецировать точки. В нашем случае каждый вектор представляет собой 2 координаты (так как 2 оси) - (v<sub>x</sub>, v<sub>y</sub>). Спроецируем наши точки (см. рис. 3 и 4) на оси. На разных осях расстояния между проекциями точек будет разное. Чем больше расстояние между точками, тем больше дисперсия (мера разброса). На ЭЭГ большая дисперсия - это либо сигнал с большой амплитудой, либо сигнал с тяжелыми хвостами. 

Формально говоря, тот вектор, при проекции на который наши данные (точки) имеют наибольшую дисперсию, "локализует на себе" самый высокоамплитудный сигнал (например тот, который пришёл не от мозга, а от сокращения мышцы). В случае большого числа каналов вектора, проекции на которые дают самые низкие дисперсии, тоже "мусор" - белый шум. Нам нужна "золотая середина".


### *Как описать математически то, что мы изобразили графически и обобщить на случай N-мерного пространства (когда электродов много)?*

Итак, мы хотим найти такие векторы (оси нашего "эллипса данных"), при проекции на которые дисперсия нашего сигнала (расстояния между спроецированными точками на оси) будет максимальной и каждый последующий вектор будет по 90 градусов к предыдущему (это 2 главных критерия PCA - максимум дисперсии и ортогональность, все алгоритмы PCA это соблюдают). Первый вектор "заберет на себя" максимальную дисперсию, второй чуть меньшую. Если электрода 2 (2 оси) - вектора всего тоже 2 (две разные компоненты). Если электродов будет 64, то это будет 64-мерное пространство (это визуализировать уже нереально, это описывается только в матричном виде (**учите линал!**) и у нас будет 64 разных вектора (64 главных компоненты - Principle components).

### *Что мы получили?*

Первая компонента (вектор 1) забрала на себя большую дисперсию, то есть ту часть сигнала, которая соответствовала максимальной амплитуде (наибольшее отклонение от среднего) - это наше высокоамплитудное моргание. Вторая компонента забрала на себя в большей степени наш "полезный сигнал" (альфа-ритм). 

Из наших исходных данных (матрица, где каждый её элемент - одна координата одной точки) мы получаем проекции (это тоже матрица, получившаяся при умножении исходной матрицы на координаты векторов). Мы можем занулить ту компоненту (столбец матрицы), которая соответствует морганию, и оставить только наш альфа-ритм. Таким образом мы избавимся от "мусора". 

Когда мы проецировали точки на вектор, мы умножали матрицу наших точек на этот вектор. Теперь после зануления, чтобы превратить проеции точек (точки на оси) обратно в облако, нам лишь нужно умножить матрицу проекций точек на обратный вектор.

Итоговая схема такая: 

нашли оси, вдоль который сильнее всего различается дисперсия -> спроецировали на них -> удалили ненужные компоненты (столбцы в матрице) -> спроецировали обратно (превратили проецкию на линии обратно в объёмное облако)

<div style="display: flex; justify-content: center; gap: 20px;">
    <div style="text-align: center;">
        <a href="assets/Proj_1.png">
            <img src="assets/Proj_1.png" width="500" style="max-width: none;">
            <br>
            <b>Рис. 3.</b> Проекция на ось 1 (PC1 - Principle component 1, забирает на себя большую дисперсию)
            <a href="assets/Proj_1.png">
        </a>
    </div>
    <div style="text-align: center;">
        <a href="assets/Proj_2.png">
            <img src="assets/Proj_2.png" width="500" style="max-width: none;">
            <br>
            <b>Рис. 4.</b> Проекция на ось 2 (PC2 - Principle component 2, забирает на себя меньшую дисперсию)
    </div>
</div>



In [1]:
# а теперь сам код)
import os 
import matplotlib
import numpy as np
import mne 
mne.viz.set_browser_backend('qt')

Using qt as 2D backend.


In [2]:
# Создаём два канала, на каждом по 4 замера
channel_1 = np.array([3,5,1,7])
channel_2 = np.array([4,2,8,6])

In [48]:
# Делаем из каналов единую матрицу
matrix = np.vstack((channel_1, channel_2)).T # "vstack" соединил два канала и сделал из каждого вектора строку в матрице. "T" - транспонирование
print(matrix)

[[3 4]
 [5 2]
 [1 8]
 [7 6]]


In [49]:
# Ищем среднее, чтобы центрировать данные. В какой бы четверти координат изначально не находилось облако данных, нам нужно сместить его в 0 
mean_matrix = np.array([np.mean(matrix[:,0]), np.mean(matrix[:,1])])
print(mean_matrix)

[4. 5.]


In [ ]:
# Центрируем матрицу
matrix_centered = matrix-mean_matrix
print(f"Центрированная матрица:\n{matrix_centered}")

# Находим матрицу ковариации - матрицу, которая описывает наше облако точек
cov_matrix = np.cov((matrix_centered).T)
print(f"\nМатрица ковариации:\n{cov_matrix}")
# В ковариационной матрице по диагонали стоят дисперсии соответствующих каналов (например на пересечении 1-1 дисперсия первого канала)
# Все остальные элементы матрицы - ковариации компонентов. Например, на пересечениях 1-2 и 2-1 стоит значение ковариации 1 и 2 каналов.

# Математически расчёт матрицы ковариации данных выглядит так:
((matrix_centered.T)@matrix_centered)/((matrix_centered.shape)[0]-1) # фактически, мы просто умножаем нашу матрицу на неё же транспонированную

Центрированная матрица:
[[-1. -1.]
 [ 1. -3.]
 [-3.  3.]
 [ 3.  1.]]

Матрица ковариации:
[[ 6.66666667 -2.66666667]
 [-2.66666667  6.66666667]]


array([[ 6.66666667, -2.66666667],
       [-2.66666667,  6.66666667]])

In [ ]:
# Расчет собственных значений (eigenvalues) и собственных векторов (eigenvectors): в начале мы этого понятия не ввели, линейная алгебра чуть ниже.
eigenvalues, eigenvectors = np.linalg.eigh(cov_matrix)
print(f"Матрица собственных значений\n{eigenvalues}")
print(f"\nСобственные векторы\n{eigenvectors}")

Матрица собственных значений
[4.         9.33333333]

Собственные векторы
[[-0.70710678 -0.70710678]
 [-0.70710678  0.70710678]]


 ----------------------------------------------------------------------------------------------------------------------------------------------

 *Умножение на любую матрицу - преобразование, которое меняет геометрию пространства 
(если это не понятно, советую посмотреть канал 3Blue1Brown, конкретно плейлист по линейной алгебре)*

##### Собственный вектор (для некоего преобразования) - вектор, которые не меняет своего направления (длину менять может) при умножении на данную матрицу. В контексте PCA: векторы, на которые мы проецируем наши данные - это собственные векторы для нашей матрицы данных.
 
##### Собственное значение - число, показывающее во сколько раз меняется длина нашего вектора при умножении на данную матрицу. Математически можно доказать, что собственное значение - это дисперсия наших спроецированных на данную ось значений.
(*Это достаточно логично, ведь чем больше разброс наших данных при проекции на данный вектор, тем более длинный вектор получается*)

In [51]:
# Проецируем нашего облако данных на каждый из найденных векторов: 
projection_2 = matrix_centered@(eigenvectors[:,0]) # PC2 - компоненты с меньшей дисперсией (собственное значение = 4)
projection_1 = matrix_centered@(eigenvectors[:,1]) # PC1 - компонента с большей дисперсией (собственное значение = 9,33)
print(f"PC1 с большей дисперсией\n{projection_1}")
print(f"\nPC2 с меньшей дисперсией\n{projection_2}")

# Специально сначала записал 2, потом 1, чтобы именно первая компонента забрала на себя большую дисперсию

PC1 с большей дисперсией
[ 0.         -2.82842712  4.24264069 -1.41421356]

PC2 с меньшей дисперсией
[ 1.41421356  1.41421356  0.         -2.82842712]


In [52]:
# Зануляем PC1 ("обнуляем моргание")
projection_1 = np.zeros_like(projection_1)
print(f"PC1\n{projection_1}")
print(f"\nPC2\n{projection_2}")

PC1
[0. 0. 0. 0.]

PC2
[ 1.41421356  1.41421356  0.         -2.82842712]


In [53]:
# Комбинируем проекции в общую матрицу
proj = np.vstack((projection_2, projection_1)).T # проекции в обратном порядке специально, чтобы четко умножилось на соответствующие обратные вектора (можете проверить)

# Ищем обратные вектора (чтобы проекции превратить обратно в облако точек)
eigenvectors_matrix = eigenvectors # в данном случае мы транспонируем, а не ищем обратную матрицу только потому, что для ортогональных векторов обратный и траспонированный вектор - одно и то же

# Превращаем проекции в облако точек
result = proj@eigenvectors_matrix
print(f"Центрированная матрица после зануления PC1\n{result}")




Центрированная матрица после зануления PC1
[[-1. -1.]
 [-1. -1.]
 [ 0.  0.]
 [ 2.  2.]]


In [54]:
# Децентрируем матрицу, возвращая нашему облаку точек "исходную позицию"
decentered = result + mean_matrix
print(f"Децентрированная матрица после зануления PC1\n{decentered}")
print (f"\nЭто и есть наш очищенный от моргания сигнал (каждое значение - вольтаж на одном канале в один момент времени)")

Децентрированная матрица после зануления PC1
[[3. 4.]
 [3. 4.]
 [4. 5.]
 [6. 7.]]

Это и есть наш очищенный от моргания сигнал (каждое значение - вольтаж на одном канале в один момент времени)
